## Spark Session

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Crypto Price Analysis").getOrCreate()

25/04/30 16:25:16 WARN Utils: Your hostname, Zwanes-MacBook.local resolves to a loopback address: 127.0.0.1; using 192.168.110.71 instead (on interface en0)
25/04/30 16:25:16 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/30 16:25:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Preprocessing

In [2]:
from pyspark.sql.functions import to_date, col, sum as _sum, when, year, month, avg, round, desc

from pyspark.sql.window import Window
from pyspark.sql.functions import first, last


In [3]:
btc_df = spark.read.csv("data/bitcoin.csv", header=True, inferSchema=True)
doge_df = spark.read.csv("data/doge.csv", header=True, inferSchema=True)
eth_df = spark.read.csv("data/eth.csv", header=True, inferSchema=True)
solana_df = spark.read.csv("data/solana.csv", header=True, inferSchema=True)
usdc_df = spark.read.csv("data/usdcoin.csv", header=True, inferSchema=True)

#### Checking Missing Values

In [4]:
def check_missing_values(df, name="DataFrame"):
    print(f"\nMissing values in {name}:")
    df.select([
        _sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in df.columns
    ]).show()

In [5]:
check_missing_values(btc_df, "Bitcoin")
check_missing_values(doge_df, "Doge Coin")
check_missing_values(eth_df, "Ethereum")
check_missing_values(solana_df, "Solana")
check_missing_values(usdc_df, "USD Coin")


Missing values in Bitcoin:
+---+----+------+----+----+---+----+-----+------+---------+
|SNo|Name|Symbol|Date|High|Low|Open|Close|Volume|Marketcap|
+---+----+------+----+----+---+----+-----+------+---------+
|  0|   0|     0|   0|   0|  0|   0|    0|     0|        0|
+---+----+------+----+----+---+----+-----+------+---------+


Missing values in Doge Coin:
+---+----+------+----+----+---+----+-----+------+---------+
|SNo|Name|Symbol|Date|High|Low|Open|Close|Volume|Marketcap|
+---+----+------+----+----+---+----+-----+------+---------+
|  0|   0|     0|   0|   0|  0|   0|    0|     0|        0|
+---+----+------+----+----+---+----+-----+------+---------+


Missing values in Ethereum:
+---+----+------+----+----+---+----+-----+------+---------+
|SNo|Name|Symbol|Date|High|Low|Open|Close|Volume|Marketcap|
+---+----+------+----+----+---+----+-----+------+---------+
|  0|   0|     0|   0|   0|  0|   0|    0|     0|        0|
+---+----+------+----+----+---+----+-----+------+---------+


Missing v

#### Checking Data Types

In [6]:
btc_df = btc_df.withColumn("Date", to_date(col("Date"), "yyyy-MM-dd HH:mm:ss"))
doge_df = doge_df.withColumn("Date", to_date(col("Date"), "yyyy-MM-dd HH:mm:ss"))
eth_df = eth_df.withColumn("Date", to_date(col("Date"), "yyyy-MM-dd HH:mm:ss"))
solana_df = solana_df.withColumn("Date", to_date(col("Date"), "yyyy-MM-dd HH:mm:ss"))
usdc_df = usdc_df.withColumn("Date", to_date(col("Date"), "yyyy-MM-dd HH:mm:ss"))

btc_df.printSchema()
doge_df.printSchema()
eth_df.printSchema()
solana_df.printSchema()
usdc_df.printSchema()


root
 |-- SNo: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Symbol: string (nullable = true)
 |-- Date: date (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Open: double (nullable = true)
 |-- Close: double (nullable = true)
 |-- Volume: double (nullable = true)
 |-- Marketcap: double (nullable = true)

root
 |-- SNo: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Symbol: string (nullable = true)
 |-- Date: date (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Open: double (nullable = true)
 |-- Close: double (nullable = true)
 |-- Volume: double (nullable = true)
 |-- Marketcap: double (nullable = true)

root
 |-- SNo: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Symbol: string (nullable = true)
 |-- Date: date (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Open: double (nullable = true

In [7]:
btc_df = btc_df.orderBy("Date")
doge_df = doge_df.orderBy("Date")
eth_df = eth_df.orderBy("Date")
solana_df = solana_df.orderBy("Date")
usdc_df = usdc_df.orderBy("Date")

### Bitcoin

#### Analisis Tren Harga

In [8]:
btc_monthly = btc_df.withColumn("Year", year("Date")).withColumn("Month", month("Date"))

btc_avg_close = btc_monthly.groupBy("Year", "Month").agg(round(avg("Close"), 2).alias("Average Close Price"))

print("\n--- 5 Data teratas dari rata-rata harga penutupan Bitcoin per bulan: ---")
btc_avg_close.orderBy("Year", "Month").show(5)
print("\n--- 5 Data terbawah dari rata-rata harga penutupan Bitcoin per bulan: ---")
btc_avg_close.orderBy(desc("Year"), desc("Month")).show(5)


--- 5 Data teratas dari rata-rata harga penutupan Bitcoin per bulan: ---
+----+-----+-------------------+
|Year|Month|Average Close Price|
+----+-----+-------------------+
|2013|    4|             141.77|
|2013|    5|             119.99|
|2013|    6|             107.76|
|2013|    7|              90.51|
|2013|    8|             113.91|
+----+-----+-------------------+
only showing top 5 rows


--- 5 Data terbawah dari rata-rata harga penutupan Bitcoin per bulan: ---
+----+-----+-------------------+
|Year|Month|Average Close Price|
+----+-----+-------------------+
|2021|    7|           34234.45|
|2021|    6|           35845.15|
|2021|    5|           46443.29|
|2021|    4|           57206.72|
|2021|    3|           54998.01|
+----+-----+-------------------+
only showing top 5 rows



In [9]:
print("\n--- Rata-rata tertinggi untuk penutupan harga---")
btc_avg_close.orderBy(desc("Average Close Price")).show(1)

print("\n--- Rata-rata terendah untuk penutupan harga---")
btc_avg_close.orderBy("Average Close Price").show(1)


--- Rata-rata tertinggi untuk penutupan harga---
+----+-----+-------------------+
|Year|Month|Average Close Price|
+----+-----+-------------------+
|2021|    4|           57206.72|
+----+-----+-------------------+
only showing top 1 row


--- Rata-rata terendah untuk penutupan harga---
+----+-----+-------------------+
|Year|Month|Average Close Price|
+----+-----+-------------------+
|2013|    7|              90.51|
+----+-----+-------------------+
only showing top 1 row



#### Analisis Tren Volatilitas

In [13]:
# Harian
btc_vol_df = btc_df.withColumn("Volatility", col("High") - col("Low"))

# Bulanan
btc_vol_monthly = btc_vol_df.withColumn("Year", year("Date")).withColumn("Month", month("Date")).groupBy("Year", "Month").agg(round(avg("Volatility"), 2).alias("Avg Monthly Volatility"))

In [14]:
# Tertinggi
print("\n--- Rata-rata volatilitas tertinggi bulanan untuk Bitcoin: ---")
btc_vol_monthly.orderBy(desc("Avg Monthly Volatility")).show(1)

# Terendah
print("\n--- Rata-rata volatilitas terendah bulanan untuk Bitcoin: ---")
btc_vol_monthly.orderBy("Avg Monthly Volatility").show(1)


--- Rata-rata volatilitas tertinggi bulanan untuk Bitcoin: ---
+----+-----+----------------------+
|Year|Month|Avg Monthly Volatility|
+----+-----+----------------------+
|2021|    5|               4340.51|
+----+-----+----------------------+
only showing top 1 row


--- Rata-rata volatilitas terendah bulanan untuk Bitcoin: ---
+----+-----+----------------------+
|Year|Month|Avg Monthly Volatility|
+----+-----+----------------------+
|2013|    8|                  4.61|
+----+-----+----------------------+
only showing top 1 row



In [ ]:
from pyspark.sql.functions import first, last, round, col, year

btc_year_df = btc_df.withColumn("Year", year("Date"))

# Hitung harga open dan close tiap tahun
btc_open_close = btc_year_df.groupBy("Year").agg(round(first("Close"), 2).alias("Open"),round(last("Close"), 2).alias("Close"))

# Hitung pertumbuhan per tahun dalam persen
btc_growth = btc_open_close.withColumn(
    "Annual Growth (%)", round(((col("Close") - col("Open")) / col("Open")) * 100, 2)
)

btc_growth.orderBy("Year").show()

+----+--------+--------+-----------------+
|Year|    Open|   Close|Annual Growth (%)|
+----+--------+--------+-----------------+
|2013|  144.54|  754.01|           421.66|
|2014|   771.4|  320.19|           -58.49|
|2015|  314.25|  430.57|            37.02|
|2016|  434.33|  963.74|           121.89|
|2017|  998.33| 14156.4|          1318.01|
|2018| 13657.2|  3742.7|            -72.6|
|2019| 3843.52|  7193.6|            87.16|
|2020| 7200.17|29001.72|           302.79|
|2021|29374.15|34235.19|            16.55|
+----+--------+--------+-----------------+

